## MARS-S2L — automatic background image selection [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UNEP-IMEO-MARS/marss2l/blob/main/notebooks/examples/background_image_selection.ipynb)

* **Author**: MARS team
---

## Overview

The MBMP retrieval compares a *target* Sentinel-2/Landsat overpass against a clear-sky **background image** of the same location from a different date. Picking a good background — same location, cloud-free, spectrally similar to the target — is what makes the retrieval clean.

This notebook shows how `BackgroundImageSelector` finds that background **directly from Google Earth Engine** (no database). It:

1. builds a target image from a known tile,
2. queries GEE for candidate backgrounds and filters them locally (cloud masks computed on the fly),
3. ranks the candidates by similarity and shows the **top-X most similar** backgrounds,
4. uses the best one for the MBMP retrieval.

This replaces the hand-picked `background_image_tile` used in `download_and_inference.ipynb` with a computed, explainable choice.

### Install `marss2l`

```bash
pip install marss2l
```

## 1. Initialize GEE

In [ ]:
import ee

ee.Authenticate()
ee.Initialize(project="YOUR-PROJECT-ID")

from marss2l.utils import setup_stream_logger

logger = setup_stream_logger()

## 2. Build the target image

We start from a known location and a target scene (the same Sentinel-2 example as `download_and_inference.ipynb`). `Location` mirrors the marsml monitoring-site object; `S2LLocationImage` mirrors `MarsLocationImage`.

In [ ]:
from marss2l.mars_sentinel2.location_image import Location, S2LLocationImage
from marss2l.mars_sentinel2.background import BackgroundImageSelector

lat, lon = 32.16492, -102.13013
tile = "S2B_MSIL1C_20250529T172859_N0511_R055_T13SGR_20250529T210525"

location = Location.from_lon_lat(lon=lon, lat=lat, margin_meters=2000, location_name="example")
target = S2LLocationImage.from_tile(tile, location=location, logger=logger)
target.tile, target.satellite, target.tile_date

## 3. Create the selector and download the target pixels

In [ ]:
selector = BackgroundImageSelector(method_bg_image="most_similar", logger=logger)
selector.download_image(target)
print(f"target percentage_clear = {target.percentage_clear:.1f}%  observability = {target.observability}")

## 4. Find the top-5 most similar background candidates

`query_background_images` queries GEE and downloads the cloud mask for a bounded set of candidates (so it can filter on the locally computed cloud fraction). `filter_and_sort_background_images` applies the cloud / satellite / orbit / date filters, and `background_images_most_similar_sorted` ranks the survivors by spectral similarity to the target.

In [ ]:
candidates = selector.query_background_images(target)
candidates = selector.filter_and_sort_background_images(target, candidates)
ranked = selector.background_images_most_similar_sorted(target, candidates, top=5)

for bg, difference in ranked:
    print(f"{bg.satellite} {bg.day}  difference = {difference * 100:.2f}%")

## 5. Plot RGB | difference | MBMP, sorted from most to least similar

In [ ]:
fig = selector.plot_all_differences(target, ranked)

## 6. Use the best background for the MBMP retrieval

The most similar candidate is `ranked[0][0]`. Equivalently, `selector.compute_background_image(target)` returns it directly. From here the retrieval is identical to `download_and_inference.ipynb`.

In [ ]:
from marss2l.mars_sentinel2 import mixing_ratio_methane

background = ranked[0][0]
selector.download_image(background)

mbmp = mixing_ratio_methane.ratio_IL(
    target.image,
    background.image,
    b12_index=selector.band_index(target, "B12"),
    b11_index=selector.band_index(target, "B11"),
    b12_index_bg=selector.band_index(background, "B12"),
    b11_index_bg=selector.band_index(background, "B11"),
    validmask=selector.validmask(target),
    validmask_bg=selector.validmask(background),
    normalize=True,
)
mbmp